# Python for Neuroscience (2026)

# 04. Patch-clamp: analysis of postsynaptic events

<!-- <img src="Images/patch-clamp_brain_slices.png" width="800"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/patch-clamp_brain_slices.png" width="800">

Synaptic transmission is the main mechanism that enables the communication between neurons through chemical signals called neurotransmitters. Generally, action potentials start at the axon initial segment of presynaptic neurons and travel along the axon, releasing neurotransmitters into the synaptic gap. These neurotransmitters then trigger chemical and electrical signals in the dendrites and cell bodies of postsynaptic neurons.

The high temporal precision of the whole-cell patch-clamp technique can give an informative readout of synaptic transmission by recording **postsynaptic currents** (ions moving through channels) and **potentials** (electrical potential differences between the intracellular and the extracellular space).

<!-- <img src="Images/04_postsynaptic events.png" width="600"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/04_postsynaptic events.png" width="600">

**GOAL**: Analyze postsynaptic currents using the scipy **[FindPeaks](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html)** function.

There are also recent Python libraries for the analysis of postsynaptic potentials:
- [minis](https://github.com/dervinism/minis). *Software for electrophysiological data analysis*. Reference: [Dervinis and Major, 2024](https://www.biorxiv.org/content/10.1101/2022.03.20.485046v6), [Dervinis and Major, 2024](https://www.frontiersin.org/journals/cellular-neuroscience/articles/10.3389/fncel.2025.1590157/full).
- [miniML](https://github.com/delvendahl/miniML). *A deep learning framework for synaptic event detection*. Reference: [O'Neill et al., 2025](https://elifesciences.org/articles/98485).
- [stimfit](https://github.com/neurodroid/stimfit). *Free, fast and simple program for viewing and analyzing electrophysiological data.*. Reference: [Guzman et al., 2014](https://www.frontiersin.org/journals/neuroinformatics/articles/10.3389/fninf.2014.00016/full).

**References and resources**
* [Find_Peaks from Scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html#scipy.signal.find_peaks)
* [A Very Short Course on Time Series Analysis](https://bookdown.org/rdpeng/timeseriesbook/) by Roger D. Peng.

# Import the libraries

In [ ]:
!pip install pyabf

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import gridspec

# Open pCLAMP abf files
import pyabf

# Scipy
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

# Statistics
from scipy.stats import ks_2samp

# Interactive plots in Jupyter lab
# %matplotlib widget
plt.close('all')

# Create the paths

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Create a folder name for the patch-clamp notebooks
notebook_name = 'patch-clamp'

# Current working directory (default is '/content' in Colab)
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):

    !wget -O "Example Data/pfc_sst_epscs.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/pfc_sst_epscs.abf"

    !wget -O "Example Data/pfc_sst_epsps.abf" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/pfc_sst_epsps.abf"
else:
    print("Download data from 'Example Data' folder.")

# Load the data

<!-- <img src="Images/04_SST_EPSCs.png" width="600"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/04_SST_EPSCs.png" width="600">

**Example data**: Whole-cell voltage-clamp recording from a somatostatin interneuron in L2/3 of the mouse PFC. Neuron was held at -70 mV to record the spontaneous excitatory postsynaptic currents (EPSCs).
- filename: **pfc_sst_epscs.abf**

In [ ]:
# ABF files
filename = 'pfc_sst_epscs'

abf = pyabf.ABF(f"{paths['data']}/{filename}.abf")
print(abf)

# In case you have several sweeps/channels, select the right one
abf.setSweep(sweepNumber=0, channel=0)

# Define the variables and sampling frequency
time = abf.sweepX  # in seconds
current = abf.sweepY
fs = int(abf.dataPointsPerMs *1000)

# Quick plot
plt.plot(time, current)
plt.show()

# Pre-processing the trace

Depending on the goal and the recording, pre-processing or processing of time series data might include: detrending, filtering and/or smoothing.

## Detrending
In detrending, we try to remove some aspects that are distorting the signal and we assume (or evaluate) are not part of our biological variable of interest. For example: rundown due to intracellular solution, photobleaching, baseline instability, etc. Thus, it has be logically related to the external confound you would like to remove. Some useful functions:

* Subtract the average (zeroing): E.g. `x - np.mean(x)`, median, median of lowest values, etc
* [Numpy Differencing](https://numpy.org/doc/stable/reference/generated/numpy.diff.html)
* [Numpy polynomial](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html)
* [Scipy exponential decay](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)
* [Scipy interpolate](https://docs.scipy.org/doc/scipy/reference/interpolate.html)
* [Scipy linear detrend](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.detrend.html)
* [Statsmodels lowess](https://www.statsmodels.org/devel/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html)

## Filtering

A filter operates on digitized data to either pass or reject a defined frequency range. All filters distort the signal to some degree, and each one has different attenuation limitations and artifacts.

Common filters for **time-domain** biological data include:
* **Bessel** filter. [Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.bessel.html)
* **Gaussian** filter. [Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter1d.html)

Common filters for **frequency-domain** applications include:
* **Butterworth** filter (non-constant delay characteristics). [Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html).

Apply the filter to the signal: `filtfilt` (zero-phase filtering). [Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.filtfilt.html)

**References**
* Widmann et al., 2015. [Digital filter design for electrophysiological data – a practical approach](https://www.sciencedirect.com/science/article/pii/S0165027014002866)
* Filters: When, Why, and How (Not) to Use Them. [Cheveigne and Nelken, 2019](https://doi.org/10.1016/j.neuron.2019.02.039)


## Smoothing

Smoothing removes certain frequencies or noise of a time series to get the general trend. Basically, it averages points with their neighbors in a time series. Two of the most common approaches are:

* Moving average. Values within the sliding window are averaged and slide through the [time-series pandas rolling](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html).
* Kernel smoothing. Instead of a time window, weights of the Kernel function are used for averaging the sliding window. It can be implemented using pandas `rolling` and Scipy `win_type`. Different kernel functions lead to different results. For example, a kernel with a shape of a gaussian. [Documentation](https://docs.scipy.org/doc/scipy/reference/signal.windows.html).

Since you need to apply a time window for smoothing, there will be missing data at the beginning and end of your time series due to the rolling window operations. Some functions allow you to handle this, in others you need to apply padding (adding zeros or a specific value to the beginning or end of the time series to make all sequences the same length.)

## Adjust the baseline

* A basic baseline adjustement can be done by subtracting a fix value or the mean of part or the full trace.

In [ ]:
# Subtract mean or median of the full trace
signal_adjusted = current - np.median(current)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(current, color='gray')
ax.plot(signal_adjusted, color='orange', alpha=0.5)
plt.show()

## Q: Subtract the 25th percentile

**Tip**: Use [np.percentile](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html).

In [ ]:
# Subtract mean or median of the full trace
signal_adjusted = current - np.percentile(current, 25)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(current, color='gray')
ax.plot(signal_adjusted, color='orange', alpha=0.5)
plt.show()

<details>

<summary><strong>Solution</strong></summary>

```python

# Subtract mean or median of the full trace
signal_adjusted = current - np.percentile(signal_raw, 25)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(current, color='gray')
ax.plot(signal_adjusted, color='orange', alpha=0.5)
plt.show()

```

# Filter the trace

Filters can be classified based on how the signal is passed or rejected at the cutoff frequencies (Figure):

- Lowpass. The filter passes frequencies below the cutoff frequency.
- Highpass. The filter passes frequencies above the cutoff frequency..
- Bandpass. A combination of low-pass and high-pass filters that passes a specific range of frequencies.
- Stop-band (notch) filters. Remove narrow frequency bands, such as 50 or 60 Hz line noise.

<!-- <img src="Images/04_Filter types.png" width="700"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/04_Filter types.png" width="700">

**Figure**. Filter types when the input is white noise (flat power spectrum). On the y-axis, log σ2 (square-root variance) represents the power of the signal. Source: “Patch clamping” by Areles Molleman.


* More information about Scipy filtering: https://docs.scipy.org/doc/scipy/reference/signal.html#filtering
* IIR notch filter: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.iirnotch.html

In [ ]:
# Adjust the signal
signal_adjusted = current - np.median(current)

# Filter parameters
lowpass_type = 'bessel'
lowpass_freq = 2000

notch_type = 'IIR'
notch_freq = 60

# Lowpass Bessel filter
lowpass_filter = signal.bessel(4,     # Order of the filter
                               lowpass_freq,  # Cutoff frequency
                               'low', # Type of filter
                               analog=False,  # Analog or digital filter
                               norm='phase',  # Critical frequency normalization
                               fs=fs,   # fs: sampling frequency
                               output='sos')

# Notch filter
b, a = signal.iirnotch(notch_freq, 30, fs)
notch_filter = signal.tf2sos(b, a)  # Convert to SOS

# Apply sequential zero-phase filtering
signal_filtered = signal.sosfiltfilt(lowpass_filter, signal_adjusted)
signal_filtered = signal.sosfiltfilt(notch_filter, signal_filtered)

# Plot the raw trace
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_title("Raw data")
ax.plot(time, signal_adjusted, linewidth=0.5, label='raw')
ax.plot(time, signal_filtered, color='orange', linewidth=0.5, label='filtered')

# Add labels and legend
ax.set_ylabel("Current (pA)")
ax.set_xlabel("Time (s)")
ax.legend()

# Adjust layout and show the plot
fig.tight_layout()
plt.show()

## Q: Calculate and plot the "baseline noise"

**Tip**: Use np.percentile.

In [ ]:
# Calculate standard deviation of the full trace
std_trace = np.std(signal_filtered)

# For example, calculate the 99th percentile for positive values and 10th percentile for negative values
baseline_positive_mean = np.percentile(signal_filtered[signal_filtered > 0], 99)
baseline_negative_mean = np.percentile(signal_filtered[signal_filtered < 0], 10)

# Create a baseline signal where values outside the range are set to NaN
baseline_signal = np.where((signal_filtered <= baseline_positive_mean) & (signal_filtered >= baseline_negative_mean),
                           signal_filtered, np.nan)

# Calculate standard deviation of the baseline signal (ignoring NaN values)
std_baseline = np.nanstd(baseline_signal)

# Plot the raw trace
fig, ax = plt.subplots(figsize=(10, 4))

# Plot the filtered and baseline trace
ax.set_title("Filtered data")
ax.plot(time, signal_filtered, color='orange')
ax.plot(time, baseline_signal, color='gray', linewidth=0.5, alpha=0.5, label='Baseline')
ax.legend()

# Add labels and legend
ax.set_ylabel("Current (pA)")
ax.set_xlabel("Time (s)")
ax.legend(frameon=False, handlelength=3, handleheight=0.7)
ax.set_xlim(0, 10)

# Adjust layout and show the plot
fig.tight_layout()
plt.show()

# Print standard deviation results
print("Standard deviation of the full trace:", std_trace)
print("Mean negative value of the baseline:", baseline_negative_mean)
print("Standard deviation of the trace baseline:", std_baseline)


<details>

<summary><strong>Solution</strong></summary>

```python

# Calculate standard deviation of the full trace
std_trace = np.std(signal_filtered)

# Calculate the 99th percentile for positive values and 10th percentile for negative values
baseline_positive_mean = np.percentile(signal_filtered[signal_filtered > 0], 99)
baseline_negative_mean = np.percentile(signal_filtered[signal_filtered < 0], 10)

# Create a baseline signal where values outside the range are set to NaN
baseline_signal = np.where((signal_filtered <= baseline_positive_mean) & (signal_filtered >= baseline_negative_mean),
                           signal_filtered, np.nan)

# Calculate standard deviation of the baseline signal (ignoring NaN values)
std_baseline = np.nanstd(baseline_signal)

# Plot the raw trace
fig, ax = plt.subplots(figsize=(10, 4))

# Plot the filtered and baseline trace
ax.set_title("Filtered data")
ax.plot(time, signal_filtered, color='orange')
ax.plot(time, baseline_signal, color='gray', linewidth=0.5, alpha=0.5, label='Baseline')
ax.legend()

# Add labels and legend
ax.set_ylabel("Current (pA)")
ax.set_xlabel("Time (s)")
ax.legend(frameon=False, handlelength=3, handleheight=0.7)
ax.set_xlim(0, 10)

# Adjust layout and show the plot
fig.tight_layout()
plt.show()

# Print standard deviation results
print("Standard deviation of the full trace:", std_trace)
print("Mean negative value of the baseline:", baseline_negative_mean)
print("Standard deviation of the trace baseline:", std_baseline)

```


# Analyze synaptic events with scipy 'find_peaks'

Check scipy find peaks documentation:
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.peak_prominences.html#scipy.signal.peak_prominences

Two important considerations of the Find Peaks function:
* It only works for positive peaks so for inward currents, use the symbol '-' in front of the data variable to convert to possitive values.
* The FindPeaks function does not consider the time variable, only the recorded data points. To convert the values to time series, divide the results by the sampling rate (s) or by sampling rate/1000 (ms).

**Tips**. For example, you can use 5 SD of baseline as reasonable starting point for event threshold. In most cases, [prominence](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.peak_prominences.html) is more useful because it detects how much a peak stands out from the surrounding peaks.

## Detection of EPSCs

- height: Min and max thresholds to detect peaks.
- threshold: Min and max vertical distance to neighboring samples.
- distance: Min horizontal distance between peaks.
- prominence: Vertical distance between the peak and lowest contour line.
- width: Min required width (in bins). E.g. For 10Khz, 10 bins = 1 ms.
- wlen: Window length to calculate prominence.
- rel_height: Relative height at which the peak width is measured.

In [ ]:
# Define the signal variable: current or voltage (raw or processed)
peaks_signal = signal_filtered  # trace after pre-processing

# Set parameters of the Find peaks function
thresh_min = 8
thresh_max = 120
thresh_prominence = 15
thresh_min_width = 0.5 * (fs/1000)

# Event window
pretrigger_window = abf.dataPointsPerMs  # Pre-event time window in ms
posttrigger_window = 4 * abf.dataPointsPerMs  # Post-event time window in ms

try:
    # Find peaks function
    peaks, peaks_dict = find_peaks(-peaks_signal,
               height=(thresh_min, thresh_max),
               threshold=None,
               distance=None,
               prominence=thresh_prominence,
               width=thresh_min_width,
               wlen=None,
               rel_height=0.5,
               plateau_size=None)

    if len(peaks) == 0:
        raise ValueError("No peaks detected in the trace")

    # Create table with results
    table = pd.DataFrame()
    table['event'] = np.arange(1, len(peaks) + 1)
    table['peak_index'] = peaks
    table['peak_time_s'] = peaks / fs
    table['event_window_start'] = (peaks_dict['left_ips'] - pretrigger_window).astype(int)
    table['event_window_end']   = (peaks_dict['right_ips'] + posttrigger_window).astype(int)
    table['peak_amp'] = peaks_dict['peak_heights']
    table['width_ms'] = peaks_dict['widths'] / (fs/1000)
    table['rise_half_amp_ms'] = (peaks - peaks_dict['left_ips'])/(fs/1000)
    table['decay_half_amp_ms'] = (peaks_dict['right_ips'] - peaks)/(fs/1000)

    # Plotting
    fig, ax = plt.subplots(figsize=(15,3))

    # Plot: Detected events in the trace
    ax.set_title("Events detection")
    ax.plot(peaks_signal)
    ax.plot(peaks, peaks_signal[peaks], "r.")
    for i, txt in enumerate(table.event):
        ax.annotate(table.event[i], (peaks[i], peaks_signal[peaks][i]))
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Current (pA)")
    ax.axes.set_xlim(10000, 20000)  # OptionaL: Zoom in the trace

except Exception as e:
    print(e)

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_EPSCs_results.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_EPSCs_results.csv", index=False)

# Show graph and table
plt.show()
table.iloc[11:25]  # Adapt to the time window if needed

## Q: Calculate ISI and Area and add to the table

**Tips**:
- You can use [np.diff](https://numpy.org/doc/stable/reference/generated/numpy.diff.html) for ISI.
- For the area, you need to iterate through the table `for i, event in table.iterrows():`, and establish beginning and start to then integrate the current charge.

In [ ]:
# Define the signal variable: current or voltage (raw or processed)
peaks_signal = signal_filtered  # trace after pre-processing

# Set parameters of the Find peaks function
thresh_min = 8
thresh_max = 120
thresh_prominence = 15
thresh_min_width = 0.5 * (fs/1000)

# Event window
pretrigger_window = abf.dataPointsPerMs  # Pre-event time window in ms
posttrigger_window = 4 * abf.dataPointsPerMs  # Post-event time window in ms

try:
    # Find peaks function
    peaks, peaks_dict = find_peaks(-peaks_signal,
               height=(thresh_min, thresh_max),
               threshold=None,
               distance=None,
               prominence=thresh_prominence,
               width=thresh_min_width,
               wlen=None,
               rel_height=0.5,
               plateau_size=None)

    if len(peaks) == 0:
        raise ValueError("No peaks detected in the trace")

    # Create table with results
    table = pd.DataFrame()
    table['event'] = np.arange(1, len(peaks) + 1)
    table['peak_index'] = peaks
    table['peak_time_s'] = peaks / fs
    table['event_window_start'] = (peaks_dict['left_ips'] - pretrigger_window).astype(int)
    table['event_window_end']   = (peaks_dict['right_ips'] + posttrigger_window).astype(int)
    table['peak_amp'] = peaks_dict['peak_heights']
    table['width_ms'] = peaks_dict['widths'] / (fs/1000)
    table['rise_half_amp_ms'] = (peaks - peaks_dict['left_ips'])/(fs/1000)
    table['decay_half_amp_ms'] = (peaks_dict['right_ips'] - peaks)/(fs/1000)

    # CALCULATE ISI
    # table['isi_s'] = # COMPLETE CODE
    table['isi_s'] = np.diff(peaks, axis=0,
                             prepend=peaks[0]) / fs  # or (peaks[1:] - peaks[:-1]) / fs

    # CALCULATE AREA
    for i, event in table.iterrows():
        individual_event = peaks_signal[int(event.event_window_start) : int(event.event_window_end)]
        table.loc[i, 'area_pA/ms'] = abs(round(individual_event.sum()/(fs/1000), 2))  # pA * ms

    # Plotting
    fig, ax = plt.subplots(figsize=(15,3))
    gridspec = fig.add_gridspec(ncols=2, nrows=1, width_ratios=[2, 1])

    # Plot 1: Detected events in the trace
    ax.set_title("Events detection")
    ax.plot(peaks_signal)
    ax.plot(peaks, peaks_signal[peaks], "r.")
    for i, txt in enumerate(table.event):
        ax.annotate(table.event[i], (peaks[i], peaks_signal[peaks][i]))
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Current (pA)")
    ax.axes.set_xlim(0, 10000)  # OptionaL: Zoom in the trace

except Exception as e:
    print(e)

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_EPSCs_results.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_EPSCs_results.csv", index=False)

# Show graph and table
plt.show()
table.iloc[0:11]  # Adapt to the time window if needed

<details>

<summary><strong>Solution</strong></summary>

```python


# Define the signal variable: current or voltage (raw or processed)
peaks_signal = signal_filtered  # trace after pre-processing

# Set parameters of the Find peaks function
thresh_min = 8
thresh_max = 120
thresh_prominence = 15
thresh_min_width = 0.5 * (fs/1000)

# Event window
pretrigger_window = abf.dataPointsPerMs  # Pre-event time window in ms
posttrigger_window = 4 * abf.dataPointsPerMs  # Post-event time window in ms

try:
    # Find peaks function
    peaks, peaks_dict = find_peaks(-peaks_signal,
               height=(thresh_min, thresh_max),  
               threshold=None,
               distance=None,
               prominence=thresh_prominence,  
               width=thresh_min_width,
               wlen=None,
               rel_height=0.5,
               plateau_size=None)
    
    if len(peaks) == 0:
        raise ValueError("No peaks detected in the trace")

    # Create table with results    
    table = pd.DataFrame()
    table['event'] = np.arange(1, len(peaks) + 1)
    table['peak_index'] = peaks
    table['peak_time_s'] = peaks / fs
    table['event_window_start'] = (peaks_dict['left_ips'] - pretrigger_window).astype(int)
    table['event_window_end']   = (peaks_dict['right_ips'] + posttrigger_window).astype(int)
    table['peak_amp'] = peaks_dict['peak_heights']
    table['width_ms'] = peaks_dict['widths'] / (fs/1000)
    table['rise_half_amp_ms'] = (peaks - peaks_dict['left_ips'])/(fs/1000)
    table['decay_half_amp_ms'] = (peaks_dict['right_ips'] - peaks)/(fs/1000)
    
    # CALCULATE ISI
    table['isi_s'] = np.diff(peaks, axis=0, prepend=peaks[0]) / fs  # or (peaks[1:] - peaks[:-1]) / fs
    
    # CALCULATE AREA
    for i, event in table.iterrows():
        individual_event = peaks_signal[int(event.event_window_start) : int(event.event_window_end)]
        table.loc[i, 'area_pA/ms'] = abs(round(individual_event.sum()/(fs/1000), 2))  # pA * ms

    # Plotting
    fig, ax = plt.subplots(figsize=(15,3))
    gridspec = fig.add_gridspec(ncols=2, nrows=1, width_ratios=[2, 1])

    # Plot 1: Detected events in the trace
    ax.set_title("Events detection")   
    ax.plot(peaks_signal)
    ax.plot(peaks, peaks_signal[peaks], "r.")
    for i, txt in enumerate(table.event):
        ax.annotate(table.event[i], (peaks[i], peaks_signal[peaks][i]))
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Current (pA)")
    ax.axes.set_xlim(0, 10000)  # OptionaL: Zoom in the trace
    
except Exception as e:
    print(e)

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_EPSCs_results.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_EPSCs_results.csv", index=False)

# Show graph and table
plt.show()
table.iloc[0:11]  # Adapt to the time window if needed

```

# Save analysis parameters

You can create a dictionary to store the main analysis settings, like sampling rate, filter types, and spike detection thresholds, in a single file.
If you save the file (.npy or .txt), then you can reuse the parameters later or keep a record of what settings were used for each analysis.

In [ ]:
analysis_parameters = {
    'sampling_frequency_Hz': fs,
    'lowpass_filter_type': lowpass_type,
    'lowpass_cutoff_Hz': lowpass_freq,
    'notch_filter_type': notch_type,
    'notch_frequency_Hz': notch_freq,
    'thresh_min': thresh_min,
    'thresh_max': thresh_max,
    'thresh_prominence': thresh_prominence,
    'thresh_min_width': thresh_min_width
}

# Option A: Save to npy file
np.save(f"{paths['analysis']}/{filename}_params", analysis_parameters)

# Option B: Save to txt file
txt = open(f"{paths['analysis']}/{filename}_params.txt","w")
txt.write(str(analysis_parameters))
txt.close()

In [ ]:
# LOAD PARAMETERS
# allow_pickle=True because we saved a list of dictionaries
events_params = np.load(f"{paths['analysis']}/{filename}_params.npy", allow_pickle=True)
events_params

# Statistics: cumulative histograms

This is a simple example of how to calculate two summary statistics and do cumulative histograms using the parameter Peak Amplitude.

In [ ]:
# Plot settings
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

# Histogram data (use other variable as needed)
hist_data = table.peak_amp

# Bin size
bin_size = 2 # Bin size for the histograms

# Calculate the number of bins required
num_bins = int(np.ceil((max(hist_data) - min(hist_data)) / bin_size))

# Plot the histogram
ax1.hist(hist_data, bins=num_bins, range=(min(hist_data), max(hist_data)))
ax1.set_title('Histogram')
ax1.set_ylabel("Count")
ax1.set_xlabel("Peak Amplitude (pA)")

# Cumulative histogram
ax2.hist(hist_data, bins=len(np.unique(hist_data)),  # Length of unique values
         range=(min(hist_data), max(hist_data)),
         histtype = 'step',
         cumulative=True,
         density=True)

# Plot the cumulative histogram
ax2.set_title('Cumulative histogram')
ax2.set_ylabel("Cumulative probability")
ax2.set_xlabel("Peak Amplitude (pA)")
fig.tight_layout()

# Summary statistics
events = len(table.event)
time_s = len(time) / fs
mean_frequency = events / time_s
mean_amplitude = np.average(hist_data)
std_amplitude = np.std(hist_data)

# Create a pandas DataFrame
summary_stats = pd.DataFrame({
    'events': [events],
    'duration': [time_s],
    'mean_frequency': [mean_frequency],
    'mean_amplitude': [mean_amplitude],
    'std_amplitude': [std_amplitude]
})

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_{hist_data.name}_hist.png", dpi=300)
summary_stats.to_csv(f"{paths['analysis']}/{filename}_{hist_data.name}_hist.csv", index=False)

plt.show()
summary_stats

## Q: Cumulative histogram against mock data and K-S test

[Kolmogorov-Smirnov test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kstest.html). Note: The test is sensitive to any differences in the two distributions. Substantial differences in shape, spread or median will result in a small P value. In contrast, the Mann-Whitney test is mostly sensitive to changes in the median.

**Tips**:
- Use ks_2samp: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html

In [ ]:
# Mock peak amplitudes
np.random.seed(10)
mock_peak_amp = np.random.uniform(low=10, high=90, size=600)
mock_data = pd.DataFrame({'peak_amp': mock_peak_amp})

# Plot cumulative histogram (fig, ax)


# hist_data ("control"): cumulative histogram
ax.hist(hist_data,
        bins=len(np.unique(hist_data)),
        range=(min(hist_data), max(hist_data)),
        histtype='step',
        cumulative=True,
        density=True,
        label='control')

# Mock data ("treatment"): cumulative histogram
# COMPLETE CODE


# KS test
# COMPLETE CODE

# Labels
ax.legend()
ax.set_title('Cumulative histogram')
ax.set_xlabel('Peak Amplitude (pA)')
ax.set_ylabel('Cumulative probability')
plt.show()

<details>

<summary><strong>Solution</strong></summary>

```python

# Mock peak amplitudes
np.random.seed(10)
mock_peak_amp = np.random.uniform(low=10, high=90, size=600)
mock_data = pd.DataFrame({'peak_amp': mock_peak_amp})

# Plot cumulative histogram
fig, ax = plt.subplots(figsize=(5,5))

# Mock data ("treatment")
ax.hist(mock_data.peak_amp,
        bins=len(np.unique(mock_data.peak_amp)),
        range=(min(mock_data.peak_amp), max(mock_data.peak_amp)),
        histtype='step',
        cumulative=True,
        density=True,
        label='treatment')

# Control: hist_data
ax.hist(hist_data,
        bins=len(np.unique(hist_data)),
        range=(min(hist_data), max(hist_data)),
        histtype='step',
        cumulative=True,
        density=True,
        label='control')

# KS test
ks_stat, p_value = ks_2samp(hist_data, mock_data.peak_amp)

print(f"KS statistic: {ks_stat:.3f}, p-value: {p_value:.4f}")

ax.legend()
ax.set_title('Cumulative histogram')
ax.set_xlabel('Peak Amplitude (pA)')
ax.set_ylabel('Cumulative probability')
plt.show()
```

In [ ]:

# Mock peak amplitudes
np.random.seed(10)
mock_peak_amp = np.random.uniform(low=10, high=90, size=600)
mock_data = pd.DataFrame({'peak_amp': mock_peak_amp})

# Plot cumulative histogram
fig, ax = plt.subplots(figsize=(5,5))

# Mock data ("treatment")
ax.hist(mock_data.peak_amp,
        bins=len(np.unique(mock_data.peak_amp)),
        range=(min(mock_data.peak_amp), max(mock_data.peak_amp)),
        histtype='step',
        cumulative=True,
        density=True,
        label='treatment')

# Control: hist_data
ax.hist(hist_data,
        bins=len(np.unique(hist_data)),
        range=(min(hist_data), max(hist_data)),
        histtype='step',
        cumulative=True,
        density=True,
        label='control')

# KS test
ks_stat, p_value = ks_2samp(hist_data, mock_data.peak_amp)

print(f"KS statistic: {ks_stat:.3f}, p-value: {p_value:.4f}")

ax.legend()
ax.set_title('Cumulative histogram')
ax.set_xlabel('Peak Amplitude (pA)')
ax.set_ylabel('Cumulative probability')
plt.show()

# Open group project: Analysis of postsynaptic potentials

- Use one of the data examples ending with `epsps`: **pfc_sst_epsps.abf**.
- Pre-process the trace: filtering, baseline adjustment/detrending.
- Detect peaks using scipy.find_peaks or numpy.
- Create a table with event properties: amplitude, timepoints, width, rise time, decay time, etc.
- Add onset and offset kinetics to the peak using exponential fitting. **Tips**: notebook D4.1 and [exponential fit](https://swharden.com/blog/2020-09-24-python-exponential-fit/). Optional: fit each peak rise and decay to an bi-exponential function.
- Create a fig, axes with 3 plots using: entire trace (ax1), shorter segment (ax2), and single detected EPSP (ax3). To adjust sizes: `fig.add_gridspec(nrows=2, ncols=2, height_ratios=[1, 1], width_ratios=[2, 1])`
- BONUS: Animate the plots. **Tips**: [matplotlib slider](https://matplotlib.org/stable/gallery/widgets/slider_demo.html), [matplotlib buttons](https://matplotlib.org/stable/gallery/widgets/buttons.html).
- BONUS: Improve the detection method by changing scipy.find_peaks parameters or using additional parameters (e.g., first derivative, [np.diff](https://numpy.org/doc/stable/reference/generated/numpy.diff.html)).
- BONUS: You can also compare the results with other packages such as [miniML](https://github.com/delvendahl/miniML).

<details>

<summary><strong>Tips</strong></summary>

```python

# Define the bi‑exponential model
def biexp_decay(x, b0, b1, tau1, b2, tau2):
    return b0 + b1 * np.exp(-x/tau1) + b2 * np.exp(-x/tau2)

# p0 = initial guess for parameters: (b0, b1, tau1, b2, tau2)

# Fit bi‑exponential
popt, pcov = curve_fit(biexp_decay, x, y, p0=p0)

```